# Blocklist Transform Example

This notebook demonstrates how to use the blocklist transform to annotate documents based on blocklisted domains.

## Overview

The blocklist transform identifies documents from blocklisted domains by:
1. Reading domain blocklists from specified files
2. Extracting domains from document URLs
3. Adding an annotation column indicating which documents are from blocklisted domains

This is useful for data quality control and content filtering in data preparation pipelines.

## Installation

**Note:** These uv pip installs need to be adapted to use the appropriate release level. 

Alternatively, the venv running the jupyter lab could be pre-configured with a requirements file that includes the right release. 

**Example for transform developers working from git clone:**
```bash
make venv
source venv/bin/activate
uv pip install jupyterlab
venv/bin/jupyter lab
```

In [19]:
%%capture
## This is here as a reference only
# Users and application developers must use the right tag for the latest from pypi
%uv pip install "data-prep-toolkit-transforms[blocklist]==1.1.5"

## Import Required Modules

In [20]:
from dpk_blocklist.runtime import Blocklist

## Configure and Run the Transform

### Parameters

The blocklist transform accepts the following key parameters:

* **input_folder**: Path to the input parquet files
* **output_folder**: Path where annotated parquet files will be written
* **blocklist_blocked_domain_list_path**: Path to directory containing domain blocklist files (files matching pattern `domains*`)
* **blocklist_annotation_column_name**: Name of the column to add with blocklist annotations (default: `"blocklisted"`)
* **blocklist_source_url_column_name**: Name of the column containing source URLs (default: `"title"`)

For a full list of parameters, please see the [README](./README.md).

### Example: Basic Blocklist Annotation

This example shows how to annotate documents using a blocklist of gambling and phishing domains:

In [21]:
Blocklist(
    input_folder="test-data/input",
    output_folder="output",
    blocklist_blocked_domain_list_path="test-data/domains/arjel",
    blocklist_annotation_column_name="blocklisted",
    blocklist_source_url_column_name="title"
).transform()

21:29:56 INFO - data factory blocklist_ Missing local configuration
21:29:56 INFO - data factory blocklist_ max_files -1, n_sample -1
21:29:56 INFO - data factory blocklist_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
21:29:56 INFO - data factory blocklist_ Data Access:  DataAccessLocal
21:29:56 INFO - pipeline id pipeline_id
21:29:56 INFO - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'commit_hash': 'UNDEFINED', 'path': 'UNDEFINED'}
21:29:56 INFO - data factory data_ max_files -1, n_sample -1
21:29:56 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
21:29:56 INFO - data factory data_ Data Access:  DataAccessLocal
21:29:56 INFO - orchestrator blocklist started at 2025-10-15 21:29:56
21:29:56 INFO - Number of files is 1, source profile {'max_file_size': 0.000718116760

0

## Verify the Output

The output folder will contain the annotated parquet files. Let's check what was created:

In [22]:
import glob
glob.glob("output/*")

['output\\metadata.json', 'output\\test1.parquet']

## Inspect the Results

Let's read the output parquet file and examine the annotations:

In [23]:
import pyarrow.parquet as pq
import pandas as pd

# Read the output parquet file
output_files = glob.glob("output/*.parquet")
if output_files:
    table = pq.read_table(output_files[0])
    df = table.to_pandas()
    print(f"\nOutput table has {len(df)} rows and {len(df.columns)} columns")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nSample data:")
    print(df.head(10))
    
    # Show blocklist statistics
    blocklisted_count = (df['blocklisted'] != '').sum()
    total_count = len(df)
    print(f"\n=== Blocklist Statistics ===")
    print(f"Total documents: {total_count}")
    print(f"Blocklisted documents: {blocklisted_count}")
    print(f"Clean documents: {total_count - blocklisted_count}")
    print(f"Blocklist rate: {blocklisted_count/total_count*100:.1f}%")
    
    if blocklisted_count > 0:
        print(f"\n=== Blocklisted Domains Found ===")
        print(df[df['blocklisted'] != ''][['title', 'blocklisted']])


Output table has 7 rows and 2 columns

Columns: ['title', 'blocklisted']

Sample data:
                               title    blocklisted
0                      https://poker          poker
1                   https://poker.fr       poker.fr
2              https://poker.foo.bar  poker.foo.bar
3                https://abc.efg.com               
4   http://asdf.qwer.com/welcome.htm               
5  http://aasdf.qwer.com/welcome.htm               
6         https://zxcv.xxx/index.asp               

=== Blocklist Statistics ===
Total documents: 7
Blocklisted documents: 3
Clean documents: 4
Blocklist rate: 42.9%

=== Blocklisted Domains Found ===
                   title    blocklisted
0          https://poker          poker
1       https://poker.fr       poker.fr
2  https://poker.foo.bar  poker.foo.bar


## Check Transform Metadata

The transform also produces a metadata.json file with processing statistics:

In [24]:
import json

metadata_file = "output/metadata.json"
try:
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    print("Transform Metadata:")
    print(json.dumps(metadata, indent=2))
except FileNotFoundError:
    print(f"Metadata file not found at {metadata_file}")

Transform Metadata:
{
  "pipeline": "pipeline_id",
  "job details": {
    "job category": "preprocessing",
    "job name": "blocklist",
    "job type": "pure python",
    "job id": "job_id",
    "start_time": "2025-10-15 21:29:56",
    "end_time": "2025-10-15 21:29:56",
    "status": "success"
  },
  "code": {
    "github": "UNDEFINED",
    "build-date": "UNDEFINED",
    "commit_hash": "UNDEFINED",
    "path": "UNDEFINED"
  },
  "job_input_params": {
    "blocked_domain_list_path": "test-data/domains/arjel",
    "annotation_column_name": "blocklisted",
    "source_url_column_name": "title",
    "checkpointing": false,
    "max_files": -1,
    "random_samples": -1,
    "files_to_use": [
      ".parquet"
    ],
    "num_processors": 0
  },
  "execution_stats": {
    "cpus": 6.0,
    "gpus": 0,
    "memory": 20.16,
    "object_store": 0,
    "execution time, min": 0.0
  },
  "job_output_stats": {
    "source_files": 1,
    "source_size": 753,
    "result_files": 1,
    "result_size": 1107

## Example: Using Multiple Blocklist Sources

You can combine multiple domain blocklists by placing multiple `domains*` files in the same directory:

In [25]:
# This example would use all domain files in test-data/domains/gambling/
# which includes domains, domains.9309, and domains.24733

Blocklist(
    input_folder="test-data/input",
    output_folder="output-gambling",
    blocklist_blocked_domain_list_path="test-data/domains/gambling",
    blocklist_annotation_column_name="gambling_domain",
    blocklist_source_url_column_name="title"
).transform()

21:30:38 INFO - data factory blocklist_ Missing local configuration
21:30:38 INFO - data factory blocklist_ max_files -1, n_sample -1
21:30:38 INFO - data factory blocklist_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
21:30:38 INFO - data factory blocklist_ Data Access:  DataAccessLocal
21:30:38 INFO - pipeline id pipeline_id
21:30:38 INFO - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'commit_hash': 'UNDEFINED', 'path': 'UNDEFINED'}
21:30:38 INFO - data factory data_ max_files -1, n_sample -1
21:30:38 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
21:30:38 INFO - data factory data_ Data Access:  DataAccessLocal
21:30:38 INFO - orchestrator blocklist started at 2025-10-15 21:30:38
21:30:38 INFO - Number of files is 1, source profile {'max_file_size': 0.000718116760

0

## Example: Custom Column Names

You can customize both the source URL column and the annotation column names:

In [26]:
# If your input data has URLs in a column named 'url' instead of 'title',
# and you want the annotation in a column named 'blocked_domain':

Blocklist(
    input_folder="your-input-folder",
    output_folder="your-output-folder",
    blocklist_blocked_domain_list_path="path-to-domains",
    blocklist_annotation_column_name="blocked_domain",
    blocklist_source_url_column_name="url"
).transform()

21:31:08 INFO - data factory blocklist_ Missing local configuration
21:31:08 INFO - data factory blocklist_ max_files -1, n_sample -1
21:31:08 INFO - data factory blocklist_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
21:31:08 INFO - data factory blocklist_ Data Access:  DataAccessLocal
21:31:08 INFO - pipeline id pipeline_id
21:31:08 INFO - code location {'github': 'UNDEFINED', 'build-date': 'UNDEFINED', 'commit_hash': 'UNDEFINED', 'path': 'UNDEFINED'}
21:31:08 INFO - data factory data_ max_files -1, n_sample -1
21:31:08 INFO - data factory data_ Not using data sets, checkpointing False, max files -1, random samples -1, files to use ['.parquet'], files to checkpoint ['.parquet']
21:31:08 INFO - data factory data_ Data Access:  DataAccessLocal
21:31:08 INFO - orchestrator blocklist started at 2025-10-15 21:31:08
21:31:08 ERROR - No input files to process - exiting
21:31:08 INFO - Completed execut

0

## Integration with Other Transforms

The blocklist transform is often used in combination with the filter transform to remove blocklisted documents:

```python
# Step 1: Annotate with blocklist
Blocklist(
    input_folder="input",
    output_folder="annotated",
    blocklist_blocked_domain_list_path="domains"
).transform()

# Step 2: Filter out blocklisted documents
from dpk_filter.runtime import Filter
Filter(
    input_folder="annotated",
    output_folder="filtered",
    filter_criteria_list=["blocklisted = ''"],  # Keep only non-blocklisted
    filter_columns_to_drop=["blocklisted"]       # Remove the annotation column
).transform()
```

## Summary

This notebook demonstrated:
- Installing and importing the blocklist transform
- Running the transform with basic parameters
- Inspecting the annotated output
- Using custom column names and multiple blocklists
- Integrating with other transforms

For more information, see:
- [Blocklist Transform README](./README.md)
- [Data Prep Kit Documentation](https://github.com/data-prep-kit/data-prep-kit)
- [Transform Project Conventions](../../README.md)